

# Autoloader


In [0]:
from pyspark.sql.functions import *
import os
import sys
project_path = os.path.join(os.getcwd(), "..","..")
sys.path.append(project_path)
from utils.transformation import reusable


In [0]:
files = dbutils.fs.ls("abfss://bronze@lohithazureproject.dfs.core.windows.net/DimUser")
display(files)

In [0]:
df = spark.read \
    .option("pathGlobFilter", "*.parquet") \
    .parquet("abfss://bronze@lohithazureproject.dfs.core.windows.net/DimUser")

df.display()

# spark structured streaming

# DimUser

In [0]:
df_user = spark.readStream.format("cloudFiles")\
.option("cloudFiles.format", "parquet")\
.option("recursiveFileLookup", "true")\
.option("cloudFiles.schemaLocation", "abfss://silver@lohithazureproject.dfs.core.windows.net/DimUser/checkpoint")\
.option("schemaEvolutionMode", "addNewColumns")\
.option("pathGlobFilter", "*.parquet")\
.load("abfss://bronze@lohithazureproject.dfs.core.windows.net/DimUser")

df_user_obj = reusable()
df_user = df_user_obj.dropColoumns(df_user,['_rescued_data'])
df_user = df_user.dropDuplicates(["user_id"])

df_user.writeStream.format("delta")\
    .option("outputMode", "append")\
    .option("checkpointLocation", "abfss://silver@lohithazureproject.dfs.core.windows.net/DimUser/checkpoint")\
    .trigger(once=True)\
    .option('path' , "abfss://silver@lohithazureproject.dfs.core.windows.net/DimUser/data")\
    .toTable("spotify_cata.silver.DimUser")


In [0]:
# display(df_user, checkpointLocation="abfss://silver@lohithazureproject.dfs.core.windows.net/checkpoint")

# DimArtist

In [0]:
df_art = spark.readStream.format("cloudFiles")\
.option("cloudFiles.format", "parquet")\
.option("recursiveFileLookup", "true")\
.option("cloudFiles.schemaLocation", "abfss://silver@lohithazureproject.dfs.core.windows.net/DimArtist/checkpoint")\
.option("schemaEvolutionMode", "addNewColumns")\
.option("pathGlobFilter", "*.parquet")\
.load("abfss://bronze@lohithazureproject.dfs.core.windows.net/DimArtist")


df_art_obj = reusable()
df_art = df_art_obj.dropColoumns(df_art,['_rescued_data'])
df_art = df_art.dropDuplicates(["artist_id"])

df_art.writeStream.format("delta")\
    .option("outputMode", "append")\
    .option("checkpointLocation", "abfss://silver@lohithazureproject.dfs.core.windows.net/DimArtist/checkpoint")\
    .trigger(once=True)\
    .option('path' , "abfss://silver@lohithazureproject.dfs.core.windows.net/DimArtist/data")\
    .toTable("spotify_cata.silver.DimArtist")


# DimTrack

In [0]:
df_track = spark.readStream.format("cloudFiles")\
.option("cloudFiles.format", "parquet")\
.option("recursiveFileLookup", "true")\
.option("cloudFiles.schemaLocation", "abfss://silver@lohithazureproject.dfs.core.windows.net/DimTrack/checkpoint")\
.option("schemaEvolutionMode", "addNewColumns")\
.option("pathGlobFilter", "*.parquet")\
.load("abfss://bronze@lohithazureproject.dfs.core.windows.net/DimTrack")

# df_track_obj = reusable()
# df_track = df_track_obj.dropColoumns(df_track,['_rescued_data'])
# df_track = df_track.dropDuplicates(["artist_id"])

# df_track.writeStream.format("delta")\
#     .option("outputMode", "append")\
#     .option("checkpointLocation", "abfss://silver@lohithazureproject.dfs.core.windows.net/DimTrack/checkpoint")\
#     .trigger(once=True)\
#     .option('path' , "abfss://silver@lohithazureproject.dfs.core.windows.net/DimTrack/data")\
#     .toTable("spotify_cata.silver.DimTrack")


df_track = df_track.withColumn("duration_flag",when(col("duration_sec")<150,"low").when(col("duration_sec")<300,"medium").otherwise("high")).withColumn("track_name",regexp_replace(col("track_name"),"-"," ")).drop("_rescued_data")

df_track.writeStream.format("delta")\
    .option("outputMode", "append")\
    .option("checkpointLocation", "abfss://silver@lohithazureproject.dfs.core.windows.net/DimTrack/checkpoint")\
    .trigger(once=True)\
    .option('path' , "abfss://silver@lohithazureproject.dfs.core.windows.net/DimTrack/data")\
    .toTable("spotify_cata.silver.DimTrack")

# DimDate

In [0]:
df_date = spark.readStream.format("cloudFiles")\
.option("cloudFiles.format", "parquet")\
.option("recursiveFileLookup", "true")\
.option("cloudFiles.schemaLocation", "abfss://silver@lohithazureproject.dfs.core.windows.net/DimDate/checkpoint")\
.option("schemaEvolutionMode", "addNewColumns")\
.option("pathGlobFilter", "*.parquet")\
.load("abfss://bronze@lohithazureproject.dfs.core.windows.net/DimDate")


df_date_obj = reusable()
df_date = df_date_obj.dropColoumns(df_date,['_rescued_data'])
#df_date = df_date.dropDuplicates(["date_id"])

df_date.writeStream.format("delta")\
    .option("outputMode", "append")\
    .option("checkpointLocation", "abfss://silver@lohithazureproject.dfs.core.windows.net/DimDate/checkpoint")\
    .trigger(once=True)\
    .option('path' , "abfss://silver@lohithazureproject.dfs.core.windows.net/DimDate/data")\
    .toTable("spotify_cata.silver.DimDate")

# FactStream

In [0]:
df_fact = spark.readStream.format("cloudFiles")\
.option("cloudFiles.format", "parquet")\
.option("recursiveFileLookup", "true")\
.option("cloudFiles.schemaLocation", "abfss://silver@lohithazureproject.dfs.core.windows.net/FactStream/checkpoint")\
.option("schemaEvolutionMode", "addNewColumns")\
.option("pathGlobFilter", "*.parquet")\
.load("abfss://bronze@lohithazureproject.dfs.core.windows.net/FactStream")


df_fact_obj = reusable()
df_fact = df_fact_obj.dropColoumns(df_fact,['_rescued_data'])
#df_date = df_date.dropDuplicates(["date_id"])

df_fact.writeStream.format("delta")\
    .option("outputMode", "append")\
    .option("checkpointLocation", "abfss://silver@lohithazureproject.dfs.core.windows.net/FactStream/checkpoint")\
    .trigger(once=True)\
    .option('path' , "abfss://silver@lohithazureproject.dfs.core.windows.net/FactStream/data")\
    .toTable("spotify_cata.silver.FactStream")